# one way accumulators

a concept introduced by [Benaloh and de Mare](http://www.michaeldemare.com/pubs/owa.pdf).

one cool property they provide is the ability to do membership testing of a set in a space-efficient manner.

# why do they don't they just use prime-order groups?

instead of using the RSA-style composite modulus, why don't they just use a prime modulus?

this is because the security of this system is based on the RSA problem:

> Given an RSA public key (n, e) and a ciphertext C = M^e (mod n), compute M

the RSA problem is trivial when N is prime. using the notation from above:

```
Given C and N=p where p is prime.
d = e^-1 (mod p-1)
d*e = 1 (mod p-1)
=> 
C^d = M^ed = M^1 = M (mod p)
```

# why do they suggest the use of safe primes?

safe primes are primes `p` such that `p = 2*p'+1` and `p'` is also prime.

the Benaloh paper calls a modulus `N` where the two primes are safe a "rigid integer"

RSA doesn't require the modulus to be a rigid integer, so why do accumuluators? the paper states:


> The advantage of using a rigid integer `n = p*q` is that the group of squares (quadratic residues) modulo `n` that are relatively prime to `n` has the propery that it has size `n' = (p-1)/2 * (q-1)/2` and the function `e_n,y(x) = x^y (mod n)` is a permutation of this group whenever `y` and `n'` are relatively prime.
> 
> Thus, if the factorization of `n` is hidden, "random" exponentations of an element of this group are extremely unlikely to produce elements of any proper subgroup. 
> 
> This means that repeated applications of of `e_n(x, y)` are extremely unlikely to reduce the size of the domain or produce random collisions.

unorganized thoughts:

* if you know you are working with a rigid integer (N = q*p s.t. p = 2*p' + 1 and q = 2*q' + 1) discovery of any of (p,p',q,q') would lead to the factorization of N.

* the paper defines n' as (p-1)/2*(q-1)/2 which is equivalent to q'*p' using my notation here.

* if you were to take a random element that is a square mod N and coprime to N it would very likely have order n'.

* there are cases where the random square would not have order n'. for an element e = x^y (mod n), where n is a rigid integer, e does not have order n' when gcd(y, n') != 1.

    * the order of the group is n' = q'*p', so the order of an element in this group must divide  n'. that gives us possible orders of (n', p', q'). 
    
    * how do we know that n' is the most likely order?
    
    * random fact: for n = pq, an element e is only a quadratic residue modulo n if it is both a quadratic residue mod p and mod q.
    
    * random fact: euler's theorem et. al do not tell us the exact order of an element but it does narrow it down.
    
    * https://math.stackexchange.com/questions/427997/on-the-number-of-quadratic-residues-pmodpq-wherep-and-q-are-odd-primes

In [13]:
import hashlib

p, q =  227, 263 # two safe primes
N = p*q
ZN = Zmod(N)
base = ZN(2)

def md(s):
    h = hashlib.md5()
    h.update(s)
    hd = h.hexdigest()
    return int(hd, 16)

class MembershipList(object):
    
    def __init__(self):
        self.x = base
        
    def add(self, member):
        self.x = self.x^member.y
        
    def contains(self, member, proof):
        return proof^member.y == self.x
        
    @property
    def accumulated_hash(self):
        return self.x
    
class Member(object):
    
    def __init__(self, name):
        self.y = md(name)
        self.mlist = MembershipList()
        self.zj = None
    
    def accumulate(self, members):
        for member in members:
            self.mlist.add(member)
        self.zj = self.mlist.accumulated_hash
        self.mlist.add(self)
        return self.mlist.accumulated_hash
    
kelby = Member("kelby")
anna = Member("anna")

# kelby's computation of the accumulator
kelby.accumulate([anna])

# anna's computation of the accumulatr
anna.accumulate([kelby])

assert(kelby.mlist.accumulated_hash == anna.mlist.accumulated_hash)
assert(anna.mlist.contains(kelby, kelby.zj))
assert(kelby.mlist.contains(anna, anna.zj))

# security of one-way accumulators

## foundational "hard" problems

## collisions

## what if i don't use a rigid integer as the modulus?